# Chapter 4: Extraction Algorithms from Scratch

In this notebook, we implement four classical randomness extractors entirely in Python—no external modules required. We'll run each on a small example bitstream so you can see how they work.

## 1. Generate Example Bitstream

We'll use a fixed 20-bit random stream for clarity.

In [ ]:
import numpy as np

np.random.seed(0)
bitstream = np.random.choice([0,1], size=20)
print('Raw bitstream:', bitstream.tolist())

## 2. Von Neumann Extractor

Pairs of bits: if they differ (01 or 10), keep the first bit; if they are the same (00 or 11), discard both.

In [ ]:
def von_neumann(bits):
    output = []
    # process in pairs
    for i in range(0, len(bits)-1, 2):
        a, b = bits[i], bits[i+1]
        if a != b:
            output.append(a)
    return output

vn_out = von_neumann(bitstream.tolist())
print('Von Neumann output:', vn_out)
print('Length:', len(vn_out))

## 3. Elias Extractor (Run-Length Based)

We group runs of identical bits. For each run, if its length is odd, we emit one bit of that value; if it's even, we emit nothing.

In [ ]:
def elias(bits):
    output = []
    if not bits:
        return []
    # walk runs
    run_val = bits[0]
    run_len = 1
    for b in bits[1:]:
        if b == run_val:
            run_len += 1
        else:
            if run_len % 2 == 1:
                output.append(run_val)
            run_val = b
            run_len = 1
    # last run
    if run_len % 2 == 1:
        output.append(run_val)
    return output

el_out = elias(bitstream.tolist())
print('Elias output:', el_out)
print('Length:', len(el_out))

## 4. Universal Hash Extractor

We hash the entire bitstream with a fixed seed, then convert the digest to bits and take the first half.

In [ ]:
import hashlib

def universal_hash(bits, seed=b'seed', out_bits=None):
    # bits → bytes
    as_bytes = bytes(int(''.join(str(b) for b in bits[i:i+8]), 2)
                     for i in range(0, len(bits), 8))
    h = hashlib.blake2b(as_bytes, key=seed, digest_size=16).digest()
    # convert to bit list
    full_bits = [int(bit) for byte in h for bit in f"{byte:08b}"]
    # default: half the length
    take = out_bits or (len(full_bits)//2)
    return full_bits[:take]

uh_out = universal_hash(bitstream.tolist())
print('Universal Hash output:', uh_out)
print('Length:', len(uh_out))

## 5. Simplified Maurer–Wolf Extractor

We split the bitstream into 4-bit blocks and XOR each block to produce one output bit.

In [ ]:
def maurer_wolf(bits):
    output = []
    for i in range(0, len(bits), 4):
        block = bits[i:i+4]
        # XOR all bits in block
        x = 0
        for b in block:
            x ^= b
        output.append(x)
    return output

mw_out = maurer_wolf(bitstream.tolist())
print('Maurer–Wolf output:', mw_out)
print('Length:', len(mw_out))

## Summary

- **Von Neumann** removes same-bit pairs.
- **Elias** uses odd‐length runs.
- **Universal Hash** scrambles and truncates.
- **Maurer–Wolf** XOR‐compresses blocks.

Each method reduces bias and improves randomness quality.